# 05 — Draft Approval via API (Phase 1c)

Review, edit, and approve drafts through the Growth API (local or deployed).

**Prerequisites:**
- Local server running: `cd scw_js && npm run dev:growth`
- Or deployed endpoint URL
- Wallet private key for EIP-191 signing

**Workflow:**
1. Authenticate with wallet signature
2. Review pending drafts via `GET /drafts`
3. Edit content via `PUT /drafts/:id`
4. Approve/reject via `POST /drafts/:id/approve` or `/reject`
5. Check insights & performance

In [ ]:
import httpx
import json
import base64
import time
import os
from dotenv import load_dotenv
from eth_account import Account
from eth_account.messages import encode_defunct

# Load keys from growth-agent/.env
load_dotenv('../.env', override=True)

# === CONFIGURATION ===
# For local dev server:
BASE_URL = 'http://localhost:8083'
# For deployed endpoint, uncomment and set:
# BASE_URL = "https://mypersonaljscloudivnad9dy-growthapi.functions.fnc.fr-par.scw.cloud"

PRIVATE_KEY = os.environ['OWNER_ETH_PRIVATE_KEY']
account = Account.from_key(PRIVATE_KEY)
print(f'Wallet: {account.address}')

In [ ]:
def make_auth_header() -> dict:
    """Create EIP-191 signed auth header with current timestamp."""
    message = f'growth-api:{int(time.time())}'
    signable = encode_defunct(text=message)
    signed = account.sign_message(signable)
    
    token_data = {
        'address': account.address,
        'signature': '0x' + signed.signature.hex(),
        'message': message,
    }
    token = base64.b64encode(json.dumps(token_data).encode()).decode()
    return {'Authorization': f'Bearer {token}'}

def api_get(path: str) -> dict:
    r = httpx.get(f'{BASE_URL}/{path}', headers=make_auth_header())
    r.raise_for_status()
    return r.json()

def api_post(path: str, body: dict = None) -> dict:
    r = httpx.post(f'{BASE_URL}/{path}', headers=make_auth_header(), json=body)
    r.raise_for_status()
    return r.json()

def api_put(path: str, body: dict) -> dict:
    r = httpx.put(f'{BASE_URL}/{path}', headers=make_auth_header(), json=body)
    r.raise_for_status()
    return r.json()

# Quick auth test
try:
    result = api_get('drafts')
    print(f'\u2705 Auth OK — connected to {BASE_URL}')
except Exception as e:
    print(f'\u274c Connection failed: {e}')

## 1. Review Pending Drafts

In [ ]:
queue = api_get('drafts')

print(f'Pending:   {len(queue.get("drafts", []))}')
print(f'Approved:  {len(queue.get("approved", []))}')
print(f'Published: {len(queue.get("published", []))}')
print(f'Rejected:  {len(queue.get("rejected", []))}')
print()

for i, draft in enumerate(queue.get('drafts', [])):
    print(f'=== Draft {i} | {draft["channel"]} ({draft["language"]}) ===')
    print(f'ID:     {draft["id"]}')
    print(f'Source: {draft.get("source_blog_post", "n/a")}')
    print(f'Link:   {draft.get("link", "n/a")}')
    print(f'Chars:  {len(draft["content"])}')
    print('---')
    print(draft['content'])
    print()

## 2. Edit a Draft

Update the content of a specific draft by ID.

In [ ]:
# === EDIT THIS CELL ===
# Set the draft ID from the list above
DRAFT_ID = queue['drafts'][0]['id']  # or paste a specific ID string
DRAFT_ID = "draft_cb352481"
NEW_CONTENT = """
The internet was built for humans.\n\nLogins, subscriptions, payment forms.\n\nBut what happens when the primary users are AI agents?\n\nI’ve been exploring x402 — a simple idea with big implications:\n→ APIs return “402 Payment Required”\n→ agents pay automatically\n→ access is unlocked\n\nNo accounts. No subscriptions. Just machine-to-machine transactions.\n\nThis flips the model:\nFrom SaaS → to pay-per-call infrastructure.\n\nFeels like a missing primitive for agent economies.\n\nCurious if this becomes:\n- the “HTTP moment” for AI services\n- or just another crypto experiment\n\nWrote a short exploration:\nhttps://www.fretchen.eu/x402/?utm_source=mastodon&utm_campaign=growth-agent
""".strip()

# --- Apply edit ---
if NEW_CONTENT != 'Your edited post content here.':
    updated = api_put(f'drafts/{DRAFT_ID}', {'content': NEW_CONTENT})
    print(f'\u2705 Updated draft {DRAFT_ID}')
    print(f'   Chars: {len(updated["content"])}')
    print(f'   Preview: {updated["content"][:100]}...')
else:
    print('No edit applied — replace NEW_CONTENT with your edited text.')

## 3. Approve or Reject Drafts

Approve drafts with optional scheduling, or reject them.

In [ ]:
from datetime import datetime, timezone, timedelta

# === LIST DRAFT IDs TO APPROVE / REJECT ===
# Use draft IDs from the review above
APPROVE_IDS = ["draft_cb352481"]
REJECT_IDS = []  # e.g. ['some-draft-id']

# === OPTIONAL: schedule overrides (draft_id -> datetime) ===
SCHEDULE = {}

# Default: one per day starting tomorrow 09:00 UTC
base = datetime.now(timezone.utc).replace(hour=9, minute=0, second=0, microsecond=0) + timedelta(days=1)
for rank, draft_id in enumerate(APPROVE_IDS):
    if draft_id not in SCHEDULE:
        SCHEDULE[draft_id] = base + timedelta(days=rank)

# --- Show plan ---
for draft_id in APPROVE_IDS:
    t = SCHEDULE[draft_id].strftime('%a %d %b %H:%M UTC')
    print(f'  \u2705 {draft_id} \u2192 approve \u2192 \U0001f4c5 {t}')
for draft_id in REJECT_IDS:
    print(f'  \u274c {draft_id} \u2192 reject')

In [ ]:
# --- Apply decisions ---
for draft_id in APPROVE_IDS:
    sched = SCHEDULE.get(draft_id)
    body = {'scheduled_at': sched.isoformat()} if sched else None
    result = api_post(f'drafts/{draft_id}/approve', body)
    print(f'\u2705 Approved: {draft_id}')

for draft_id in REJECT_IDS:
    result = api_post(f'drafts/{draft_id}/reject')
    print(f'\u274c Rejected: {draft_id}')

# Refresh queue
queue = api_get('drafts')
print(f'\nPending:   {len(queue.get("drafts", []))}')
print(f'Approved:  {len(queue.get("approved", []))}')
print(f'Rejected:  {len(queue.get("rejected", []))}')

## 4. View Insights & Performance

In [ ]:
insights = api_get('insights')
print('=== Insights ===')
print(json.dumps(insights, indent=2, default=str))

In [ ]:
performance = api_get('performance')
print('=== Performance ===')
print(json.dumps(performance, indent=2, default=str))

## Next Steps

- Run **notebook 04** to publish approved drafts where `scheduled_at <= now`
- Once local testing passes, deploy: `cd scw_js && npm run deploy`